In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [0]:
dest_path = 'olist_dataset.bronze.'

### Category

In [0]:
cat_schema = T.StructType([
    T.StructField('category_in_brazilian', T.StringType(),True),
    T.StructField('category_in_english',T.StringType(),True)
])

In [0]:
cat_df = spark.read.csv('/Volumes/olist_dataset/source_data/raw_files/category/*.csv',
                        header= True,
                        inferSchema=True,
                        mode= 'DROPMALFORMED',
                        schema=cat_schema)

display(cat_df)

In [0]:
cat_df = cat_df.withColumn('source_file',F.col('_metadata.file_path'))\
                        .withColumn('ingested_at',F.current_timestamp())

display(cat_df.limit(4))

In [0]:
cat_df.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f'{dest_path}category')

### Customers

In [0]:
cust_schema = T.StructType([
    T.StructField('customer_id',T.StringType(), False),
    T.StructField('customer_unique_id',T.StringType(), False),
    T.StructField('zip_code', T.IntegerType(), True),
    T.StructField('city', T.StringType(), True),
    T.StructField('state',T.StringType(),True)
])

In [0]:
cust_df = spark.read.csv('/Volumes/olist_dataset/source_data/raw_files/customers/*.csv',
                        header= True,
                        inferSchema=True,
                        mode= 'DROPMALFORMED',
                        schema= cust_schema
                        )

In [0]:
cust_df = cust_df.withColumn('source_file',F.col('_metadata.file_path'))\
                        .withColumn('ingested_at',F.current_timestamp())



In [0]:
display(cust_df.limit(4))

In [0]:
cust_df.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable(f'{dest_path}customers')

### Geolocation

In [0]:
geo_schema = T.StructType([
    T.StructField('zip_code_prefix',T.IntegerType(),True),
    T.StructField('latitude',T.DoubleType(),True),
    T.StructField('longitude',T.DoubleType(),True),
    T.StructField('city',T.StringType(),True),
    T.StructField('state',T.StringType(),True)
])

In [0]:
geo_df = spark.read.csv('/Volumes/olist_dataset/source_data/raw_files/geolocation/*.csv',
                        header= True,
                        inferSchema=True,
                        mode= 'DROPMALFORMED',
                        schema=geo_schema
                        )

In [0]:
display(geo_df.limit(4))

In [0]:
geo_df = geo_df.withColumn('source_file',F.col('_metadata.file_path'))\
                        .withColumn('ingested_at',F.current_timestamp())


In [0]:
geo_df.write.format('delta')\
    .option('mergeSchema', True)\
    .mode('overwrite')\
    .saveAsTable(f'{dest_path}geolocation')

### Items

In [0]:
itm_schema = T.StructType([
    T.StructField('order_id',T.StringType(),True),
    T.StructField('item_id',T.IntegerType(),True),
    T.StructField('product_id',T.StringType(),True),
    T.StructField('seller_id',T.StringType(),True),
    T.StructField('shipping_date',T.TimestampType(),True),
    T.StructField('price',T.DecimalType(10,2),True),
    T.StructField('shipping_cost',T.DecimalType(10,2),True)
])

In [0]:
itm_df = spark.read.csv('/Volumes/olist_dataset/source_data/raw_files/items/*.csv',
                        header= True,
                        mode= 'DROPMALFORMED',
                        inferSchema= True,
                        schema= itm_schema)

In [0]:
display(itm_df.limit(4))

In [0]:
itm_df = itm_df.withColumn('source_file', F.col('_metadata.file_path'))\
    .withColumn('ingested_at', F.current_timestamp())

In [0]:
itm_df.write.format('delta')\
    .option('mergeSchema', True)\
    .mode('overwrite')\
    .saveAsTable(f'{dest_path}items')

### Orders

In [0]:
ord_schema = T.StructType([
    T.StructField('order_id',T.StringType(),False),
    T.StructField('customer_id',T.StringType(),False),
    T.StructField('status',T.StringType(),True),
    T.StructField('purchase_time', T.TimestampType(),True),
    T.StructField('approved_time',T.TimestampType(),True),
    T.StructField('delivered_carrier_date',T.TimestampType(),True),
    T.StructField('customer_delivery_date',T.TimestampType(),True),
    T.StructField('customer_delivery_est_date',T.TimestampType(),True)
])

In [0]:
ord_df = spark.read.csv('/Volumes/olist_dataset/source_data/raw_files/orders/*.csv',
                        header= True,
                        mode= 'DROPMALFORMED',
                        inferSchema= True,
                        schema= ord_schema
                        )

In [0]:
display(ord_df)

In [0]:
ord_df = ord_df.withColumn('source_file', F.col('_metadata.file_path'))\
        .withColumn('ingested_at',F.current_timestamp())

In [0]:
ord_df.write.format('delta')\
    .option('mergeSchema', True)\
    .mode('overwrite')\
    .saveAsTable(f'{dest_path}orders')